# Agent智能体
参考文档：https://hello-agents.datawhale.cc/

在人工智能领域，智能体被定义为任何能够通过传感器（Sensors）感知其所处环境（Environment），并自主地通过执行器（Actuators）采取行动（Action）以达成特定目标的实体。

## 实现最简单的智能体

```mermaid
flowchart LR
    subgraph Agent
        direction TB
        a2(LLM) <--> a4(Tool)
        a2(LLM) <--> a5(Memory)
    end
    a1(input) --> Agent
    Agent --> a3(output)

### 安装所需的库

In [1]:
#!pip install requests openai
!pip show requests openai

Name: requests
Version: 2.34.2
Summary: Python HTTP for Humans.
Home-page: 
Author: 
Author-email: Kenneth Reitz <me@kennethreitz.org>
License: Apache-2.0
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: certifi, charset_normalizer, idna, urllib3
Required-by: datasets, flashinfer-python, gguf, jupyterlab_server, mistral_common, modelscope, opentelemetry-exporter-otlp-proto-http, ray, tavily-python, tiktoken, vllm
---
Name: openai
Version: 2.37.0
Summary: The official Python library for the openai API
Home-page: https://github.com/openai/openai-python
Author: 
Author-email: OpenAI <support@openai.com>
License: Apache-2.0
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: anyio, distro, httpx, jiter, pydantic, sniffio, tqdm, typing-extensions
Required-by: vllm


tavily-python是一个强大的 AI 搜索 API 客户端，用于获取实时的网络搜索结果，可以在[官网](https://app.tavily.com/home)注册后获取 API

In [2]:
# !pip install tavily-python 
!pip show tavily-python

Name: tavily-python
Version: 0.7.25
Summary: Python wrapper for the Tavily API
Home-page: https://github.com/tavily-ai/tavily-python
Author: Tavily AI
Author-email: support@tavily.com
License: 
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: httpx, requests, tiktoken
Required-by: 


In [3]:
# 设置你的apikey
%env TAVILY_API_KEY=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

env: TAVILY_API_KEY=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


### 系统指令模版

In [36]:
AGENT_SYSTEM_PROMPT = """
你是一个智能旅行助手。你的任务是分析用户的请求，并使用可用工具一步步地解决问题。

# 可用工具:
- `get_weather(city: str)`: 查询指定城市的实时天气。
- `get_attraction(city: str, weather: str)`: 根据城市和天气搜索推荐的旅游景点。

# 输出格式要求:
你的每次回复必须严格遵循以下格式，包含一对Thought和Action：

Thought: [你的思考过程和下一步计划]
Action: [你要执行的具体行动]

Action的格式必须是以下之一！：
1. 调用工具：function_name(arg_name="arg_value")，例如：get_weather(city="北京") 。
2. 结束任务：Finish[你的最终答案内容]。

# 重要提示:
- 每次只输出一对Thought-Action
- Action必须在同一行，不要换行
- 当收集到足够信息可以回答用户问题时，必须使用 Action: Finish[你的最终答案内容] 格式结束

请开始吧！
"""

### 工具1：查询真实天气
使用免费天气api服务 wttr.in

In [5]:
import requests

def get_weather(city: str) -> str:
    """
    通过调用 wttr.in API 查询今天真实的天气信息。
    """
    print("正在调用tool:get_weathern......")
    # API端点，我们请求JSON格式的数据
    url = f"https://wttr.in/{city}?format=j1"
    
    try:
        # 发起网络请求
        response = requests.get(url)
        # 检查响应状态码是否为200 (成功)
        response.raise_for_status() 
        # 解析返回的JSON数据
        data = response.json()
            
        # 提取当前天气状况
        current_condition = data['current_condition'][0]
        weather_desc = current_condition['weatherDesc'][0]['value']
        temp_c = current_condition['temp_C']
        # print("今天天气内容：",current_condition)
        
        # 格式化成自然语言返回
        return f"{city}当前天气:{weather_desc}，气温{temp_c}摄氏度"
        
    except requests.exceptions.RequestException as e:
        # 处理网络错误
        return f"错误:查询天气时遇到网络问题 - {e}"
    except (KeyError, IndexError) as e:
        # 处理数据解析错误
        return f"错误:解析天气数据失败，可能是城市名称无效 - {e}"

In [6]:
get_weather("苏州")

正在调用tool:get_weathern......


'苏州当前天气:Partly Cloudy ，气温32摄氏度'

### 工具2：搜索并推荐旅游景点

In [7]:
import os
from tavily import TavilyClient

def get_attraction(city: str, weather: str) -> str:
    """
    根据城市和天气，使用Tavily Search API搜索并返回优化后的景点推荐。
    """
    # 1. 从环境变量中读取API密钥
    print("正在调用tool:get_attraction......")
    api_key = os.environ.get("TAVILY_API_KEY")
    if not api_key:
        return "错误:未配置TAVILY_API_KEY环境变量。"

    # 2. 初始化Tavily客户端
    tavily = TavilyClient(api_key=api_key)
    
    # 3. 构造一个精确的查询
    query = f"'{city}' 在'{weather}'天气下最值得去的旅游景点推荐及理由"
    
    try:
        # 4. 调用API，include_answer=True会返回一个综合性的回答
        response = tavily.search(query=query, search_depth="basic", include_answer=True)
        
        # 5. Tavily返回的结果已经非常干净，可以直接使用
        # response['answer'] 是一个基于所有搜索结果的总结性回答
        if response.get("answer"):
            return response["answer"]
        
        # 如果没有综合性回答，则格式化原始结果
        formatted_results = []
        for result in response.get("results", []):
            formatted_results.append(f"- {result['title']}: {result['content']}")
        
        if not formatted_results:
             return "抱歉，没有找到相关的旅游景点推荐。"

        return "根据搜索，为您找到以下信息:\n" + "\n".join(formatted_results)

    except Exception as e:
        return f"错误:执行Tavily搜索时出现问题 - {e}"

In [9]:
get_attraction("苏州", get_weather("苏州"))

正在调用tool:get_weathern......
正在调用tool:get_attraction......


"Under partly cloudy skies with a temperature of 32°C, visit the Humble Administrator's Garden for a serene experience. This historic garden offers a peaceful retreat and showcases classical Chinese garden design."

### 将工具都放在工具箱里

In [10]:
# 将所有工具函数放入一个字典，方便后续调用
available_tools = {
    "get_weather": get_weather,
    "get_attraction": get_attraction,
}

### 使用LLM推理

In [13]:
import torch
import re
import gc
from modelscope import AutoModelForCausalLM, AutoTokenizer

def CleanMemory():
    torch.cuda.empty_cache()
    gc.collect()

#### 使用Qwen/Qwen3-0.6B本地模型推理，在6_LMandLora.ipynb里已经下载过了

In [12]:
!modelscope download --model Qwen/Qwen3-0.6B --local_dir ./Qwen/Qwen3-0.6B


 _   .-')                _ .-') _     ('-.             .-')                              _ (`-.    ('-.
( '.( OO )_             ( (  OO) )  _(  OO)           ( OO ).                           ( (OO  ) _(  OO)
 ,--.   ,--.).-'),-----. \     .'_ (,------.,--.     (_)---\_)   .-----.  .-'),-----.  _.`     \(,------.
 |   `.'   |( OO'  .-.  ',`'--..._) |  .---'|  |.-') /    _ |   '  .--./ ( OO'  .-.  '(__...--'' |  .---'
 |         |/   |  | |  ||  |  \  ' |  |    |  | OO )\  :` `.   |  |('-. /   |  | |  | |  /  | | |  |
 |  |'.'|  |\_) |  |\|  ||  |   ' |(|  '--. |  |`-' | '..`''.) /_) |OO  )\_) |  |\|  | |  |_.' |(|  '--.
 |  |   |  |  \ |  | |  ||  |   / : |  .--'(|  '---.'.-._)   \ ||  |`-'|   \ |  | |  | |  .___.' |  .--'
 |  |   |  |   `'  '-'  '|  '--'  / |  `---.|      | \       /(_'  '--'\    `'  '-'  ' |  |      |  `---.
 `--'   `--'     `-----' `-------'  `------'`------'  `-----'    `-----'      `-----'  `--'      `------'


Successfully Downloaded from model Qwen/Qwen3-0.6B.


In [11]:
def load_model(model_name="./Qwen/Qwen3-0.6B"):
    # 加载分词器
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    # 加载模型
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.bfloat16,
        # dtype=torch.float16, #上面的不能用就用下面的
        device_map="auto"
    )
    return model, tokenizer

In [14]:
model, tokenizer = load_model("./Qwen/Qwen3-0.6B")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

#### 输入提示词

In [43]:
user_prompt = "你好，请帮我查询一下今天苏州的天气，然后根据天气推荐一个合适的旅游景点。"
prompt_history = [f"用户请求: {user_prompt}"]

print(f"用户输入: {user_prompt}")

用户输入: 你好，请帮我查询一下今天苏州的天气，然后根据天气推荐一个合适的旅游景点。


In [12]:
def get_qwen_output(messages, model, tokenizer):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True # 是否使用思考模式
    )
    # 编码
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    # 推理
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=32768 #最大上下文长度
    )
    # 得到输出并转 ids
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    try:
        # 找到思考结束的符号的id的位置 151668 (</think>)
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0
    # 思考内容
    thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
    # 输出内容
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
    return thinking_content, content

### Agent行动循环

In [45]:
for i in range(5): # 设置最大循环次数
    print(f"--- 循环 {i+1} ---\n")
    
    # 3.1. 构建Prompt
    full_prompt = "\n".join(prompt_history)
    # 封装到messages
    messages = [
        {'role': 'system', 'content': AGENT_SYSTEM_PROMPT},
        {'role': 'user', 'content': full_prompt}
    ]
    print("历史记录", messages)
    # 3.2. 调用LLM进行思考
    think_output, output = get_qwen_output(messages, model, tokenizer)
    print(f"模型think:\n{think_output}\n")
    # 模型可能会输出多余的Thought-Action，需要截断
    match = re.search(r'(Thought:.*?Action:.*?)(?=\n\s*(?:Thought:|Action:|Observation:)|\Z)', output, re.DOTALL)
    if match:
        truncated = match.group(1).strip()
        if truncated != output.strip():
            output = truncated
            print("已截断多余的 Thought-Action 对")
    print(f"模型输出:\n{output}\n")
    prompt_history.append(output)
    
    # 3.3. 解析并执行行动
    action_match = re.search(r"Action: (.*)", output, re.DOTALL)
    if not action_match:
        observation = "错误: 未能解析到 Action 字段。请确保你的回复严格遵循 'Thought: ... Action: ...' 的格式。"
        observation_str = f"Observation: {observation}"
        print(f"{observation_str}\n" + "="*40)
        prompt_history.append(observation_str)
        continue
    action_str = action_match.group(1).strip()

    # if action_str.startswith("Finish"):
    if "Finish" in action_str:
        final_answer = re.search(r"Finish\[(.*)\]", action_str).group(1)
        print(f"任务完成，最终答案: {final_answer}")
        break
    
    tool_name = re.search(r"(\w+)\(", action_str).group(1)
    args_str = re.search(r"\((.*)\)", action_str).group(1)
    kwargs = dict(re.findall(r'(\w+)="([^"]*)"', args_str))

    if tool_name in available_tools:
        observation = available_tools[tool_name](**kwargs)
    else:
        observation = f"错误:未定义的工具 '{tool_name}'"

    # 3.4. 记录观察结果
    observation_str = f"Observation: {observation}"
    print(f"{observation_str}\n" + "="*40)
    prompt_history.append(observation_str)

--- 循环 1 ---

历史记录 [{'role': 'system', 'content': '\n你是一个智能旅行助手。你的任务是分析用户的请求，并使用可用工具一步步地解决问题。\n\n# 可用工具:\n- `get_weather(city: str)`: 查询指定城市的实时天气。\n- `get_attraction(city: str, weather: str)`: 根据城市和天气搜索推荐的旅游景点。\n\n# 输出格式要求:\n你的每次回复必须严格遵循以下格式，包含一对Thought和Action：\n\nThought: [你的思考过程和下一步计划]\nAction: [你要执行的具体行动]\n\nAction的格式必须是以下之一！：\n1. 调用工具：function_name(arg_name="arg_value")，例如：get_weather(city="北京") 。\n2. 结束任务：Finish[你的最终答案内容]。\n\n# 重要提示:\n- 每次只输出一对Thought-Action\n- Action必须在同一行，不要换行\n- 当收集到足够信息可以回答用户问题时，必须使用 Action: Finish[你的最终答案内容] 格式结束\n\n请开始吧！\n'}, {'role': 'user', 'content': '用户请求: 你好，请帮我查询一下今天苏州的天气，然后根据天气推荐一个合适的旅游景点。'}]
模型think:
<think>
好的，用户让我查询苏州今天的天气，然后根据天气推荐旅游景点。首先，我需要确认用户的需求。用户可能是在计划去苏州的旅行，所以需要知道天气情况来安排行程。接下来，我应该使用get_weather工具来获取苏州的实时天气数据。然后，使用get_attraction工具，传入苏州和查询到的天气信息，得到推荐的景点。

首先，我需要调用get_weather函数，参数是city="苏州"。然后，得到天气数据后，再调用get_attraction，传入city="苏州"和weather参数。在调用这两个工具后，应该能获取到具体的天气情况和景点推荐，最后将结果整理成用户友好的回答。
</think>

已截断多余的 Thought-Action 对
模型输出:
Thought: 调用get_weathe

In [84]:
del model
del tokenizer
CleanMemory()

**如果结果有问题，那就对了，下面是使用LLM正确输出样例**

```
用户输入: 你好，请帮我查询一下今天北京的天气，然后根据天气推荐一个合适的旅游景点。
========================================
--- 循环 1 ---

正在调用大语言模型...
大语言模型响应成功。
模型输出:
Thought: 首先需要获取北京今天的天气情况，之后再根据天气情况来推荐旅游景点。
Action: get_weather(city="北京")

Observation: 北京当前天气:Sunny，气温26摄氏度
========================================      
--- 循环 2 ---

正在调用大语言模型...
大语言模型响应成功。
模型输出:
Thought: 现在已经知道了北京今天的天气是晴朗且温度适中，接下来可以基于这个信息来推荐一个适合的旅游景点了。
Action: get_attraction(city="北京", weather="Sunny")

Observation: 北京在晴天最值得去的旅游景点是颐和园，因其美丽的湖景和古建筑。另一个推荐是长城，因其壮观的景观和历史意义。
========================================
--- 循环 3 ---

正在调用大语言模型...
大语言模型响应成功。
模型输出:
Thought: 已经获得了两个适合晴天游览的景点建议，现在可以根据这些信息给用户提供满意的答复。
Action: Finish[今天北京的天气是晴朗的，气温26摄氏度，非常适合外出游玩。我推荐您去颐和园欣赏美丽的湖景和古建筑，或者前往长城体验其壮观的景观和深厚的历史意义。希望您有一个愉快的旅行！]

任务完成，最终答案: 今天北京的天气是晴朗的，气温26摄氏度，非常适合外出游玩。我推荐您去颐和园欣赏美丽的湖景和古建筑，或者前往长城体验其壮观的景观和深厚的历史意义。希望您有一个愉快的旅行！
```

## 智能体核心工作流（思维范式）

### ReAct和其实现

ReAct范式通过一种特殊的提示工程来引导模型，使其每一步的输出都遵循一个固定的轨迹：

- Thought (思考)： 这是智能体的“内心独白”。它会分析当前情况、分解任务、制定下一步计划，或者反思上一步的结果。
- Action (行动)： 这是智能体决定采取的具体动作，通常是调用一个外部工具，例如 Search['华为最新款手机']。
- Observation (观察)： 这是执行Action后从外部工具返回的结果，例如搜索结果的摘要或API的返回值。

智能体将不断重复这个 Thought -> Action -> Observation 的循环，将新的观察结果追加到历史记录中，形成一个不断增长的上下文，直到它在Thought中认为已经找到了最终答案，然后输出结果。

```mermaid
graph LR
    Q[用户问题] --> T1[Thought: 分析问题,决定下一步行动]
    T1 --> A[Action: 调用工具 / 搜索 / 计算]
    A --> O[Observation: 观察返回结果]
    O --> J{得到最终答案?}
    J -- 否 --> T2[Thought: 基于观察再推理]
    T2 --> A
    J -- 是 --> F[Final Answer]

#### 系统提示词

In [128]:
# ReAct 提示词模板
REACT_PROMPT_TEMPLATE = """
请注意，你是一个有能力调用外部工具的智能助手。

可用工具如下:
{tools}

**请严格按照以下格式进行回应!**:

Thought: 你的思考过程，用于分析问题、拆解任务和规划下一步行动。
Action: 你决定采取的行动，只能是以下格式中的一个:
- `{{tool}}[{{params}}]`:调用一个可用工具。
- `Finish[最终答案]`:当你认为已经获得最终答案时。
当你收集到足够的信息，能够回答用户的最终问题时，你必须在Action:字段后使用 Finish[最终答案] 来输出最终答案。

现在，请开始解决以下问题:
Question: {question}
History: {history}
"""

#### 工具与工具包

In [86]:
def search(query: str):
    print(f"🔍 正在进行网页搜索: {query}")
    return """截至2026年6月1日，英伟达最新发布的消费级桌面显卡是 GeForce RTX 50 系列（基于 Blackwell 架构），
    包括 RTX 5090、5080 等型号；专业/数据中心领域最新为 RTX PRO Blackwell（如 RTX PRO 6000）和 H200/H100 系
    列（H200 于2024年发布，仍属当前旗舰AI加速卡）。‌‌
    """

In [87]:
from typing import Dict, Any

class ToolExecutor:
    def __init__(self):
        self.tools: Dict[str, Dict[str, Any]] = {}

    def registerTool(self, name: str, description: str, func: callable):
        if name in self.tools:
            print(f"警告:工具 '{name}' 已存在，将被覆盖。")
        self.tools[name] = {"description": description, "func": func}
        print(f"工具 '{name}' 已注册。")

    def getTool(self, name: str) -> callable:
        return self.tools.get(name, {}).get("func")

    def getAvailableTools(self) -> str:
        return "\n".join([
            f"- {name}: {info['description']}" 
            for name, info in self.tools.items()
        ])

In [98]:
toolExecutor = ToolExecutor()
search_description = "一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。"
toolExecutor.registerTool("Search", search_description, search)
print("\n--- 可用的工具 ---")
print(toolExecutor.getAvailableTools())

tool_name = "Search"
tool_input = "英伟达最新GPU型号"
tool_function = toolExecutor.getTool(tool_name)
observation = tool_function(tool_input)
print(observation)

工具 'Search' 已注册。

--- 可用的工具 ---
- Search: 一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。
🔍 正在进行网页搜索: 英伟达最新GPU型号
截至2026年6月1日，英伟达最新发布的消费级桌面显卡是 GeForce RTX 50 系列（基于 Blackwell 架构），
    包括 RTX 5090、5080 等型号；专业/数据中心领域最新为 RTX PRO Blackwell（如 RTX PRO 6000）和 H200/H100 系
    列（H200 于2024年发布，仍属当前旗舰AI加速卡）。‌‌
    


#### 输出解析

In [116]:
def ReAct_parse_output(self, text: str):
    
    # 从Thought: 匹配到 Action: 或文本末尾
    thought_match = re.search(r"Thought:\s*(.*?)(?=\nAction:|$)", text, re.DOTALL)
    
    # 从Action: 匹配到文本末尾
    action_match = re.search(r"Action:\s*(.*?)$", text, re.DOTALL)
    
    thought = thought_match.group(1).strip() if thought_match else None
    action = action_match.group(1).strip() if action_match else None
    return thought, action

In [117]:
text="""Thought: 首先需要获取北京今天的天气情况，之后再根据天气情况来推荐旅游景点。
Action: Get_weather[北京]

Observation: 北京当前天气:Sunny，气温26摄氏度
"""
thought, action = ReAct_parse_output(None, text)
print("thought:", thought)
print("action:", action)

thought: 首先需要获取北京今天的天气情况，之后再根据天气情况来推荐旅游景点。
action: Get_weather[北京]

Observation: 北京当前天气:Sunny，气温26摄氏度


#### 工具解析

In [118]:
def ReAct_parse_action(self, action_text: str):

    # 从头匹配，先匹配字母数字下划线：get_weather，然后匹配 [，然后批匹配任意字符，最后匹配 ]
    # re.DOTALL :让 . 匹配包括换行符在内的所有字符
    match = re.match(r"(\w+)\[(.*)\]", action_text, re.DOTALL)
    if match:
        return match.group(1), match.group(2)
    return None, None

In [119]:
tool, params = ReAct_parse_action(None, action)
print("调用函数：",tool,"(",params,")")

调用函数： Get_weather ( 北京 )


#### ReActAgent简单实现

In [132]:
class ReActAgent:
    def __init__(self, tool_executor: ToolExecutor, model=None, max_steps=5):
        self.model = model,
        self.tool_executor = tool_executor
        self.max_steps = max_steps
        self.history = []
        
    def run(self, question: str):
        self.history = [] 
        current_step = 0
        model, tokenizer = load_model()
        while current_step < self.max_steps:
            current_step+=1
            print(f"--- 第 {current_step} 步 ---")
            
            # 准备提示词
            available_tools = self.tool_executor.getAvailableTools()
            history_str = "\n".join(self.history)
            prompt = REACT_PROMPT_TEMPLATE.format(
                tools=available_tools,
                question=question,
                history=history_str
            )
            # print("提示词：",prompt,"\n")
            
            # 模型推理
            messages = [{"role": "user", "content": prompt}]
            think_output, output = get_qwen_output(messages, model, tokenizer)
            # print(think_output, "\n" ,output)
            # 输出解析
            thought, action = self.ReAct_parse_output(output)
            print("思考过程：",thought) if thought else None
            if not action:
                print(output)
                print(thought, action)
                break
                
            if action.startswith("Finish"):
                # 如果是Finish指令，提取最终答案并结束
                final_answer = re.match(r"Finish\[(.*)\]", action).group(1)
                print(f"🎉 最终答案: {final_answer}")
                return final_answer

            # 工具解析
            tool, params = self.ReAct_parse_action(action)
            if not tool or not params:
                print(output)
                print(tool, params)
                break

            print(f"🎬 行动: {tool}[{params}]")
            
            tool_function = self.tool_executor.getTool(tool_name)
            if not tool_function:
                observation = f"错误:未找到名为 '{tool}' 的工具。"
            else:
                observation = tool_function(params) # 调用真实工具
                
            print(f"👀 观察: {observation}")
            # 将本轮的Action和Observation添加到历史记录中
            self.history.append(f"Action: {action}")
            self.history.append(f"Observation: {observation}")

        # 循环结束
        del model
        del tokenizer
        CleanMemory()
        print("已达到最大步数，流程终止。")

In [133]:
ReActAgent.ReAct_parse_output = ReAct_parse_output
ReActAgent.ReAct_parse_action = ReAct_parse_action

In [134]:
agent = ReActAgent(toolExecutor)
agent.run("英伟达最新的GPU型号是什么？")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

--- 第 1 步 ---
思考过程： 英伟达（NVIDIA）最新推出的GPU型号是A100，这是2023年发布的，目前在市场中广泛使用。
🎬 行动: Search[NVIDIA最新GPU型号]  
Finish[NVIDIA最新GPU型号是A100]
🔍 正在进行网页搜索: NVIDIA最新GPU型号]  
Finish[NVIDIA最新GPU型号是A100
👀 观察: 截至2026年6月1日，英伟达最新发布的消费级桌面显卡是 GeForce RTX 50 系列（基于 Blackwell 架构），
    包括 RTX 5090、5080 等型号；专业/数据中心领域最新为 RTX PRO Blackwell（如 RTX PRO 6000）和 H200/H100 系
    列（H200 于2024年发布，仍属当前旗舰AI加速卡）。‌‌
    
--- 第 2 步 ---
Finish[NVIDIA最新GPU型号是A100]
None None
已达到最大步数，流程终止。


**显然0.6B的小模型还是听不懂提示词，正确结果如下**

```
问题：华为最新的手机是哪个？
--- 第 1 步 ---
Thought: 要回答这个问题，我需要查找华为最新发布的手机型号及其主要特点。这些信息可能在我的现有知识库之外，因此需要使用搜索引擎来获取最新数据。
Action: Search[华为最新手机型号及主要卖点]
🤔 思考: 要回答这个问题，我需要查找华为最新发布的手机型号及其主要特点。这些信息可能在我的现有知识库之外，因此需要使用搜索引擎来获取最新数据。
🎬 行动: Search[华为最新手机型号及主要卖点]
🔍 正在执行 [SerpApi] 网页搜索: 华为最新手机型号及主要卖点
👀 观察: [1] 华为手机- 华为官网
智能手机 ; Mate 系列. 非凡旗舰 · HUAWEI Mate XTs. 非凡大师 ; Pura 系列. 先锋影像 · HUAWEI Pura 80 Pro+ ; Pocket 系列. 美学新篇. HUAWEI Pocket 2 ; nova 系列. 专业人像.

[2] 2025年华为手机哪一款性价比高？华为手机推荐与市场分析 ...
现在华为手机最大的卖点只剩下鸿蒙HarmonyOS系统，以及饱受争议的品牌信仰。 这里推荐目前值得入手的几款华为系列手机，根据不同预算自行选择:. 华为目前最受欢迎，也是搭载 ...

[3] 2025年华为新款手机哪个性价比高？10款华为新款手机推荐
选华为主要还是要推荐高端手机，Mate 70和Pura 70系列是最新发布的旗舰机型。 HUAWEI Mate 70. 优点是，拍照配置依旧顶级，全焦段覆盖，适合专业摄影，做工出色，户外抗摔 ...

--- 第 2 步 ---
Thought: 根据搜索结果，华为最新发布的旗舰机型包括Mate 70和Pura 80 Pro+。为了确定最新型号及其主要卖点，我将重点放在这些信息上。从提供的链接来看，Mate 70系列和Pura 80 Pro+都是近期发布的产品，但具体哪一个是“最新”还需要进一步确认。同时，我可以从这些信息中提取出它们的主要
卖点。
Action: Finish[根据最新信息，华为的最新手机可能是HUAWEI Pura 80 Pro+或HUAWEI Mate 70。其中，HUAWEI Mate 70的主要卖点包括顶级的拍照配置，全焦段覆盖，适合专业摄影，做工出色，并且具有良好的户外抗摔性能。而HUAWEI Pura 80 Pro+则强调了先锋影像技术。]
🤔 思考: 根据搜索结果，华为最新发布的旗舰机型包括Mate 70和Pura 80 Pro+。为了确定最新型号及其主要卖点，我将重点放在这些信息上。从提供的链接来看，Mate 70系列和Pura 80 Pro+都是近期发布的产品，但具体哪一个是“最新”还需要进一步确认。同时，我可以从这些信息中提取出它们的主要 
卖点。
🎉 最终答案: 根据最新信息，华为的最新手机可能是HUAWEI Pura 80 Pro+或HUAWEI Mate 70。其中，HUAWEI Mate 70的主要卖点包括顶级的拍照配置，全焦段覆盖，适合专业摄影，做工出色，并且具有良好的户外抗摔性能。而HUAWEI Pura 80 Pro+则强调了先锋影像技术。

### Plan-and-Solve

顾名思义，这种范式将任务处理明确地分为两个阶段：先规划 (Plan)，后执行 (Solve)。

- 规划阶段 (Planning Phase)： 首先，智能体会接收用户的完整问题。它的第一个任务不是直接去解决问题或调用工具，而是将问题分解，并制定出一个清晰、分步骤的行动计划。这个计划本身就是一次大语言模型的调用产物。
- 执行阶段 (Solving Phase)： 在获得完整的计划后，智能体进入执行阶段。它会严格按照计划中的步骤，逐一执行。每一步的执行都可能是一次独立的 LLM 调用，或者是对上一步结果的加工处理，直到计划中的所有步骤都完成，最终得出答案。

```mermaid
graph LR
    Q[复杂问题] --> P[Planner: 把问题拆解成<br/>子步骤]
    P --> S1[Step 1:执行子任务1]
    S1 --> S2[Step 2:执行子任务2]
    S2 --> SN[...]
    SN --> OUT[综合输出最终答案]

### Reflection

为智能体引入一种事后（post-hoc）的自我校正循环，使其能够像人类一样，审视自己的工作，发现不足，并进行迭代优化。

其核心工作流程可以概括为一个简洁的三步循环：执行 -> 反思 -> 优化。

- 执行 (Execution)：首先，智能体使用我们熟悉的方法（如 ReAct 或 Plan-and-Solve）尝试完成任务，生成一个初步的解决方案或行动轨迹。这可以看作是“初稿”。
- 反思 (Reflection)：接着，智能体进入反思阶段。它会调用一个独立的、或者带有特殊提示词的大语言模型实例，来扮演一个“评审员”的角色。这个“评审员”会审视第一步生成的“初稿”，并从多个维度进行评估，例如：
  - 事实性错误：是否存在与常识或已知事实相悖的内容？
  - 逻辑漏洞：推理过程是否存在不连贯或矛盾之处？
  - 效率问题：是否有更直接、更简洁的路径来完成任务？
  - 遗漏信息：是否忽略了问题的某些关键约束或方面？ 根据评估，它会生成一段结构化的反馈 (Feedback)，指出具体的问题所在和改进建议。
- 优化 (Refinement)：最后，智能体将“初稿”和“反馈”作为新的上下文，再次调用大语言模型，要求它根据反馈内容对初稿进行修正，生成一个更完善的“修订稿”。

**可以用同一个模型切换角色，节省成本；也可以是两个不同模型**

```mermaid
graph LR
    Q[任务输入] --> G[执行: 生成初版答案]
    G --> R[反思: 自我评估/批评]
    R --> J{质量达标?}
    J -- 否 --> M[优化: 根据反馈修改/重写]
    M --> R
    J -- 是 --> F[输出最终答案]

### 总结

 |   | ReAct   | 	Plan-and-Solve  | 	Reflection |
 | ---- | ---- | ---- |----|
 | 核心思路 | 	边想边做,交替进行  | 	先全盘规划,再逐步执行 | 	做完了再回头审视、修正 |
 | 关键动作 | 	Thought → Action → Observation	 | Plan → Steps → Output | 	Produce → Critique → Refine |
 | 适合场景 | 	需要调用工具/搜索/外部信息 | 	多步推理、数学、应用题 | 	需要高质量输出、自纠错 |

## 智能体搭建平台

**Coze**：https://www.coze.cn/

- 核心定位：由字节跳动推出的 Coze[1]，主打零代码/低代码的 Agent 的构建体验，让不具备编程背景的用户也能轻松创造。
- 特点分析：Coze 拥有极其友好的可视化界面，用户可以像搭建乐高积木一样，通过拖拽插件、配置知识库和设定工作流来创建智能体。其内置了极为丰富的插件库，并支持一键发布到抖音、飞书、微信公众号等多个主流平台，极大地简化了分发流程。
- 适用人群：**非技术用户**，AI 应用的入门用户、产品经理、运营人员，以及希望快速将创意变为可交互产品的个人创作者。

**Dify**

- 核心定位：Dify 是一个开源的、功能全面的 LLM 应用开发与运营平台[2]，旨在为开发者提供从原型构建到生产部署的一站式解决方案。
- 特点分析：它融合了后端服务和模型运营的理念，支持 Agent 工作流、RAG Pipeline、数据标注与微调等多种能力。对于追求专业、稳定、可扩展的企业级应用而言，Dify 提供了坚实的基础。
- 适用人群：**开发者**，有一定技术背景的开发者、需要构建可扩展的企业级 AI 应用的团队。

**n8n**

- 核心定位：n8n 本质上是一个开源工作流自动化工具[3]，而非纯粹的 LLM 平台。近年来，它积极集成了 AI 能力。
- 特点分析：n8n 的强项在于“连接”。它拥有数百个预置的节点，可以轻松地将各类 SaaS 服务、数据库、API 连接成复杂的自动化业务流程。你可以在这个流程中嵌入 LLM 节点，使其成为整个自动化链路中的一环。虽然在 LLM 功能的专一度上不如前两者，但其通用自动化能力是独一无二的。不过，其学习曲线也相对陡峭。
- 适用人群：**AI只是其中一个节点**，需要将 AI 能力深度整合进现有业务流程、实现高度定制化自动化的开发者和企业。

**通用ai智能体**

- MiniMax Agent：https://agent.minimaxi.com/
- 智谱 Agent：https://chatglm.cn/
- Trae SOLO：https://solo.trae.cn/

## 智能体框架

### 单智能体框架

**LangGraph**：
- 作为 LangChain 生态的扩展，LangGraph 另辟蹊径，把 Agent 的"思考-行动-观察"循环画成显式状态图(节点:每一步操作 + 边:跳转逻辑 + 检查点)
- 适合:复杂长任务、要可调试、要审计(金融/医疗)
- 不适合:简单对话

```mermaid
stateDiagram-v2
    direction LR
    [*] --> 思考
    思考 --> 工具调用: 决定调工具
    思考 --> [*]: 直接回答
    工具调用 --> 观察: 工具返回
    观察 --> 思考: 还要继续
    观察 --> [*]: 任务完成
    工具调用 --> 检查点: 状态持久化
    检查点 --> 工具调用
    观察 --> 人机确认: 关键步骤
    人机确认 --> 思考

**AgentScope**：
- 国产通用 Agent 框架,定位和 LangGraph 同级
- 它不仅提供了直观易用的编程接口，更重要的是内置了分布式部署、容错恢复、可观测性等企业级特性，使其特别适合构建需要长期稳定运行的生产环境应用。
- 适合:国内团队、中文场景、要单 Agent 也可能要扩展到多 Agent
- 不适合:只想要最轻量库

```mermaid
graph LR
    subgraph L1[应用层]
        A1[CoPaw 桌面 Agent]
        A2[其他应用]
    end
    subgraph L2[框架层 - AgentScope]
        RA[ReActAgent]
        DA[DialogAgent]
        MA[多 Agent 编排]
    end
    subgraph L3[能力组件]
        T[Tools 工具]
        M[ReMe 记忆]
        MCP[MCP 客户端]
    end
    subgraph L4[模型层]
        Q[Qwen]
        G[GLM]
        L[Llama]
    end
    L1 --> L2
    L2 --> L3
    L2 --> L4

```mermaid
sequenceDiagram
    autonumber
    participant U as 👤 User
    participant H as 📨 MsgHub<br/>(消息总线)
    participant A as 🤖 Agent A<br/>研究员
    participant B as 🤖 Agent B<br/>工程师
    participant C as 🤖 Agent C<br/>审核员
    participant M as 🧠 ReMe<br/>记忆
    participant T as 🔧 MCP<br/>工具

    U->>H: 投递任务:调研+实现+审核
    H->>A: 路由 → Agent A
    A->>M: 读取历史上下文
    M-->>A: 返回相关记忆
    A->>T: 🔧 调搜索/抓取工具
    T-->>A: 返回原始数据
    A->>H: 📤 投递:研究成果
    H->>B: 路由 → Agent B
    B->>M: 💾 写入:方案草稿
    B->>T: 🔧 执行代码/跑测试
    T-->>B: 测试结果
    B->>H: 📤 投递:实现+测试报告
    H->>C: 路由 → Agent C
    C->>M: 读取:待审核产物
    C->>H: ✅ 投递:审核通过
    H->>U: 🎉 最终交付

### 多智能体框架

**AutoGen**：
- AutoGen 的设计哲学根植于"以对话驱动协作"。它巧妙地将复杂的任务解决流程，映射为不同角色的智能体之间的一系列自动化对话。
- 适合:研究、需要人类参与的协作、有多角色对话需求

```mermaid
graph TB
    C((🎯<br/>AutoGen))
    C --> M1[🤝 4 协作模式<br/>GroupChat · Sequential<br/>Hierarchical · Nested]
    C --> M2[🛠️ 7 大能力<br/>人在环·代码沙箱·工具·<br/>RAG·记忆·可观测·分布式]
    C --> M3[🧠 多模型<br/>OpenAI / Anthropic /<br/>Azure / 本地 / 国产]
    C --> M4[🚪 3 入口<br/>SDK · Studio · Bench]

    classDef centerStyle fill:#7c3aed,stroke:#5b21b6,color:#fff,stroke-width:3px
    classDef modeStyle fill:#10b981,stroke:#047857,color:#fff
    classDef capStyle fill:#f59e0b,stroke:#b45309,color:#fff
    classDef modelStyle fill:#3b82f6,stroke:#1d4ed8,color:#fff
    classDef entryStyle fill:#ec4899,stroke:#9d174d,color:#fff

    class C centerStyle
    class M1 modeStyle
    class M2 capStyle
    class M3 modelStyle
    class M4 entryStyle

```mermaid
sequenceDiagram
    participant U as UserProxy
    participant A as Coder
    participant B as Reviewer
    participant C as Tester
    U->>A: 帮我写个排序函数
    A->>B: 提交代码
    B-->>A: 指出 3 个 bug
    A->>A: 修复
    A->>C: 跑测试
    C-->>U: 全过 ✅

**CAMEL**：
- 与 AutoGen 和 AgentScope 这样功能全面的框架不同，CAMEL最初的核心目标是探索如何在最少的人类干预下，让**两个智能体**通过“角色扮演”自主协作解决复杂任务。
- 两大核心概念：角色扮演 (Role-Playing) 和 引导性提示 (Inception Prompting)。
    - 一个扮演“AI 用户” (AI User)，负责提出需求、下达指令和构思任务步骤；另一个则扮演“AI 助理” (AI Assistant)，负责根据指令执行具体操作和提供解决方案。
    - “引导性提示”是在对话开始前，分别注入给两个智能体的一段精心设计的、结构化的初始指令（System Prompt）。包含自身角色、协作者角色、共同目标、行为约束和沟通协议。
- 适合:Agent 协作机制研究、自主智能体行为探索、论文复现
- 不适合:直接做产品(可观测性、稳定性、生态都不如 AutoGen)

```mermaid
sequenceDiagram
    participant SYS as 🎯 System Prompt<br/>(Inception 机制)
    participant U as User Agent<br/>(甲方/需求方)
    participant A as Assistant Agent<br/>(乙方/执行方)
    participant T as 🔧 工具/代码

    SYS->>U: 植入角色:你是任务规划者
    SYS->>A: 植入角色:你是执行者
    U->>A: 📋 派子任务 1
    A->>T: 调工具/写代码
    T-->>A: 返回结果
    A-->>U: ✅ 交付
    U->>A: 📋 派子任务 2
    A->>T: 调工具/写代码
    T-->>A: 返回结果
    A-->>U: ✅ 交付
    Note over U,A: 🔁 循环直到共同目标完成
    A-->>U: 🎉 最终交付

### **与之前的工作流（思维范式）和 OpenClaw等 的关系**

```mermaid
graph LR
    L3[📦 应用产品层<br/>OpenClaw 、 Hermes Agent 、 CoPaw 、 Claude Code 、 TRAE SOLO]
    L2[🛠 库级框架层<br/>LangChain 、 LangGraph 、 AutoGen 、 CrewAI<br/>AgentScope 、 Smolagents 、 MetaGPT 、 CAMEL]
    L1[💡 思维范式层<br/>ReAct 、 Plan-and-Solve 、 Reflection 、 Tree/Graph of Thoughts]

    L3 -->|基于| L2
    L2 -->|实现| L1

    classDef app fill:#ececff,stroke:#ec174d,color:#000
    classDef lib fill:#7ec6ff,stroke:#1d4ed8,color:#000
    classDef think fill:#fff5ad,stroke:#047857,color:#000
    class L3 app
    class L2 lib
    class L1 think

## MyAgent框架 🚧🚧🚧施工中.... 

## 记忆系统

人类记忆系统

```mermaid
graph LR
    env[环境输入]
    
    subgraph SM[感觉记忆]
        direction LR
        vis[视觉]
        hear[听觉]
        touch[触觉]
        more[......]
    end
    
    subgraph STM[短时记忆]
        direction TB
        stm[短时记忆]
        stm --> stm
    end
    
    ltm[长时记忆]
    
    f1[遗忘]
    f2[遗忘<br/>因衰退或干扰]
    f3[遗忘<br/>因干扰或提取失败]
    
    env ==> SM
    SM ==> STM
    SM --> f1
    
    STM ==>|存储| ltm
    ltm ==>|提取| STM
    
    STM --> f2
    ltm --> f3
    

    classDef envStyle fill:#a7f3d0,stroke:#059669,color:#000
    classDef memStyle fill:#fed7aa,stroke:#ea580c,color:#000
    classDef forgetStyle fill:#e5e7eb,stroke:#6b7280,color:#000
    
    class env envStyle
    class vis,hear,touch,more,stm,ltm,reh memStyle
    class f1,f2,f3 forgetStyle

**记忆系统要解决的问题**：
- 上下文丢失：在长对话中，早期的重要信息可能会因为上下文窗口限制而丢失
- 个性化缺失：Agent无法记住用户的偏好、习惯或特定需求
- 学习能力受限：无法从过往的成功或失败经验中学习改进
- 一致性问题：在多轮对话中可能出现前后矛盾的回答
- 知识时效性：大模型的训练数据有时间截止点，无法获取最新信息
- 专业领域知识：通用模型在特定领域的深度知识可能不足
- 事实准确性：通过检索验证，减少模型的幻觉问题
- 可解释性：提供信息来源，增强回答的可信度

HelloAgents记忆与RAG系统整体架构
```mermaid
graph LR
    %% 前端层
    subgraph FE[前端层]
        SA[SimpleAgent] --> TR[ToolRegistry]
        TR --> MemoryTool[MemoryTool]
        TR --> RAGTool[RAGTool]
    end  
    
    %% 管理层
    subgraph MG[管理层]
        MM[MemoryManager]
        RP[RAGPipeline]
    end
    MemoryTool --> MM
    RAGTool --> RP
    
    %% 记忆类型层
    subgraph MemT[记忆类型层]
        WM[WorkingMemory<br/>纯内存+TTL]
        EM[EpisodicMemory<br/>事件序列]
        SM[SemanticMemory<br/>知识图谱]
        PM[PerceptualMemory<br/>多模态]
    end
    MM --> WM
    MM --> EM
    MM --> SM
    MM --> PM
    
    %% RAG处理层
    subgraph RAGT[RAG处理层]
        DP[DocumentProcessor<br/>文档解析]
        QA[智能问答引擎<br/>LLM增强]
    end
    RP --> DP
    RP --> QA
    
    %% 存储抽象层
    subgraph ST[存储抽象层]
        SDS[SQLiteDocumentStore<br/>结构化存储]
        NGS[Neo4jGraphStore<br/>图谱管理]
        QVS[QdrantVectorStore<br/>向量检索]
    end
    
    %% 基础设施层
    subgraph IF[基础设施层]
        SQLite[(SQLite)]
        Neo4j[(Neo4j)]
        Qdrant[(Qdrant)]
        ES[EmbeddingService<br/>统一嵌入]
    end
    
    %% 记忆到存储
    EM --> SDS
    EM --> QVS
    SM --> NGS
    SM --> QVS
    PM --> QVS
    PM --> SDS
    
    %% RAG到存储/嵌入
    DP --> ES
    QA --> QVS
    QA --> ES
    
    %% 存储到基础设施
    SDS --> SQLite
    NGS --> Neo4j
    QVS --> Qdrant
    
    %% 纯内存标注
    WM -.->|纯内存| 内存[内存]

    %% 配色
    classDef embedStyle fill:#fef3c7,stroke:#ca8a04,color:#000
    classDef dbStyle fill:#f5f3ff,stroke:#7c3aed,color:#000
    class ES embedStyle
    class SQLite,Neo4j,Qdrant dbStyle

### 记忆系统工作流程

```mermaid
graph LR
    subgraph EP[外部信息处理]
        SI[感知输入<br/>Sensory Input] --> EN[编码<br/>Encoding]
    end
    
    subgraph MC[记忆系统核心]
        ST{存储<br/>Storage}
        CO[整合<br/>Consolidation]
        FO[遗忘<br/>Forgetting]
        RE[检索<br/>Retrieval]
    end
    
    subgraph OUT[输出]
        RB[回忆与行为输出<br/>Recall & Behavior]
    end
    
    EN --> ST
    ST -->|信息巩固与强化| CO
    ST -->|信息丢失| FO
    ST -->|信息提取| RE
    RE --> RB

    classDef purpleStyle fill:#e9d5ff,stroke:#7c3aed,color:#000
    classDef blueStyle fill:#bfdbfe,stroke:#2563eb,color:#000
    classDef orangeStyle fill:#fed7aa,stroke:#ea580c,color:#000
    classDef greenStyle fill:#bbf7d0,stroke:#16a34a,color:#000
    classDef redStyle fill:#fecaca,stroke:#dc2626,color:#000
    
    class SI,RB purpleStyle
    class EN,RE blueStyle
    class ST orangeStyle
    class CO greenStyle
    class FO redStyle

### RAG

RAG（Retrieval-Augmented Generation）是结合检索和生成的AI框架，让Agent能够从外部知识库检索相关信息，结合这些信息生成更准确的回答
- 检索器：从知识库中查找相关文档
- 生成器：基于检索结果和问题生成回答

```mermaid
sequenceDiagram
    autonumber
    actor U as 👤 用户
    participant L as 📄 Loader
    participant SP as ✂️ Splitter
    participant EM as 🔢 Embedder
    participant DB as 🗄️ Vector DB
    participant RT as 🔍 Retriever
    participant LLM as 🧠 LLM

    Note over U,DB: 📥 阶段一:PDF 入库
    U->>L: 指定 PDF 路径
    L->>L: 按页解析
    L-->>SP: Document 列表
    SP->>SP: 切分 chunk_size=500
    SP-->>EM: 切好的 chunks
    EM->>EM: chunks → 向量
    EM->>DB: 存向量 + 原文
    DB-->>U: ✅ 入库完成

    Note over U,LLM: 🔍 阶段二:RAG 问答
    U->>RT: 提问
    RT->>EM: 问题 → 向量
    EM-->>RT: 问题向量
    RT->>DB: 相似度搜索
    DB-->>RT: top-k 相关 chunks
    RT->>LLM: 拼 prompt<br/>问题 + chunks
    LLM->>LLM: 生成答案
    LLM-->>U: 🎉 最终答案

#### RAG检索

In [27]:
# !pip install langchain langchain-community langchain-text-splitters langchain-openai chromadb pypdf sentence-transformers
!pip show langchain langchain-community langchain-text-splitters langchain-openai chromadb pypdf sentence-transformers

Name: langchain
Version: 1.3.2
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: langchain-community
Version: 0.4.2
Summary: Community contributed LangChain integrations.
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: aiohttp, httpx-sse, langchain-classic, langchain-core, langsmith, numpy, pydantic-settings, pyyaml, requests, sqlalchemy, tenacity
Required-by: 
---
Name: langchain-text-splitters
Version: 1.1.2
Summary: LangChain text splitting utilities
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: langchain-core
Required-by: langchain-cl

导入所需的库

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings, HuggingFaceBgeEmbeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.embeddings import ModelScopeEmbeddings
from langchain_community.vectorstores import Chroma

**加载pdf**

In [3]:
loader = PyPDFLoader("./data/Agent/Happy-LLM-0727.pdf")
documents = loader.load()                               # 每页一个 Document 对象
print(f"📄 加载完成,共 {len(documents)} 页")

📄 加载完成,共 171 页


**切分文本**

In [4]:
# RecursiveCharacterTextSplitter 是官方推荐的"递归切分器"
# 优先按段落 → 句子 → 词切,保持语义完整

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,                                     # 每块最大 500 字符
    chunk_overlap=50,                                   # 相邻块重叠 50 字符,避免切断上下文
    separators=["\n\n", "\n", "。", "!", "?", " ", ""]  # 中文友好分隔符
)
splits = text_splitter.split_documents(documents)
print(f"✂️  切分完成,共 {len(splits)} 个块")

✂️  切分完成,共 559 个块


**Embedding嵌入器**

In [74]:
!modelscope download --model BAAI/bge-small-zh-v1.5 --local_dir ./BAAI/bge-small-zh-v1.5


 _   .-')                _ .-') _     ('-.             .-')                              _ (`-.    ('-.
( '.( OO )_             ( (  OO) )  _(  OO)           ( OO ).                           ( (OO  ) _(  OO)
 ,--.   ,--.).-'),-----. \     .'_ (,------.,--.     (_)---\_)   .-----.  .-'),-----.  _.`     \(,------.
 |   `.'   |( OO'  .-.  ',`'--..._) |  .---'|  |.-') /    _ |   '  .--./ ( OO'  .-.  '(__...--'' |  .---'
 |         |/   |  | |  ||  |  \  ' |  |    |  | OO )\  :` `.   |  |('-. /   |  | |  | |  /  | | |  |
 |  |'.'|  |\_) |  |\|  ||  |   ' |(|  '--. |  |`-' | '..`''.) /_) |OO  )\_) |  |\|  | |  |_.' |(|  '--.
 |  |   |  |  \ |  | |  ||  |   / : |  .--'(|  '---.'.-._)   \ ||  |`-'|   \ |  | |  | |  .___.' |  .--'
 |  |   |  |   `'  '-'  '|  '--'  / |  `---.|      | \       /(_'  '--'\    `'  '-'  ' |  |      |  `---.
 `--'   `--'     `-----' `-------'  `------'`------'  `-----'    `-----'      `-----'  `--'      `------'


Successfully Downloaded from model BAAI/bge-small-zh

In [6]:
embedding = HuggingFaceBgeEmbeddings(model_name="./BAAI/bge-small-zh-v1.5")

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ./BAAI/bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
vector = embedding.embed_query("什么是机器学习")
print(f"✅ 成功！向量维度: {len(vector)}")

✅ 成功！向量维度: 512


**存入Chroma向量数据库**

In [8]:
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embedding,
    # persist_directory="./data/Agent/chroma_db"          # 持久化目录,下次直接复用，  也可以只加载在内存

)
print(f"💾 入库完成,数据已存到 ./data/Agent/chroma_db")

💾 入库完成,数据已存到 ./data/Agent/chroma_db


删除数据库

In [9]:
# import shutil
# shutil.rmtree("./data/Agent/chroma_db", ignore_errors=True) 

**相似度检索**

In [10]:
query = "Agent是什么?"                          # 替换成你的问题
docs = vectorstore.similarity_search(query, k=3)        # 返回最相关的 3 块
print(f"\n🔍 查询: {query}")
print(f"📚 检索到 {len(docs)} 个相关块:\n")
for i, doc in enumerate(docs, 1):
    print(f"--- 块 {i} (来源: 第 {doc.metadata.get('page', '?') + 1} 页) ---")
    print(doc.page_content[:200])                       # 打印前 200 字


🔍 查询: Agent是什么?
📚 检索到 3 个相关块:

--- 块 1 (来源: 第 164 页) ---
个公司组织结构来完成商业策划。AutoGen、ChatDev等框架⽀持这类系统的构建。
探索与学习型Agent（Exploration & Learning Agents）：
特点： 这类Agent不仅执⾏任务，还能在与环境的交互中主动学习新知识、新技能或优化⾃身策略，类似于强
化学习中的Agent概念。
⼯作⽅式： 可能包含更复杂的记忆和反思机制，能够根据成功或失败的经验调整未来的规划和⾏动。

--- 块 2 (来源: 第 168 页) ---
Agent 的⼯作流程如下：
1. 接收⽤户输⼊。
2. 调⽤⼤模型（如 Qwen），并告知其可⽤的⼯具及其 Schema。
3. 如果模型决定调⽤⼯具，Agent 会解析请求，执⾏相应的 Python 函数。
4. Agent 将⼯具的执⾏结果返回给模型。
5. 模型根据⼯具结果⽣成最终回复。
6. Agent 将最终回复返回给⽤户。
如图7.9所示，Agent 调⽤⼯具流程：
图7.9 Age
--- 块 3 (来源: 第 162 页) ---
注：7.2 章节的所有代码均可在 Happy-LLM Chapter7 RAG 中找到。
7.3 Agent  
7.3.1 什么是 LLM Agent？  
简单来说，⼤模型Agent是⼀个以LLM为核⼼“⼤脑”，并赋予其⾃主规划、记忆和使⽤⼯具能⼒的系统。 它不再仅
仅是被动地响应⽤户的提示（Prompt），⽽是能够：
1. 理解⽬标（Goal Understanding）： 接收⼀个相对复杂


#### RAG增强生成

In [26]:
# 在简单写一个Agent里的函数
# model, tokenizer = load_model()
# think_output, output = get_qwen_output(messages, model, tokenizer)

**将我们本地的qwen3-0.6b接入langchain**

In [14]:
from langchain_core.language_models.llms import LLM
from typing import Optional, List, Any

# 编写 LangChain 适配器
class Qwen3LLM(LLM):
    model: Any
    tokenizer: Any

    @property
    def _llm_type(self) -> str:
        return "qwen3"

    def _call(self, prompt: str, stop: Optional[List[str]] = None, **kwargs) -> str:
        messages = [{"role": "user", "content": prompt}]
        think_output, output = get_qwen_output(messages, self.model, self.tokenizer)      # 调用我们的推理函数
        print(f"\n🧠 [模型思考]: {think_output}\n")                                       # 如果需要看思考过程，可以打印一下
        return output                                                                     # 只把最终输出返回给 RAG 链

In [15]:
model, tokenizer = load_model()
qwen3_06b_llm = Qwen3LLM(model=model, tokenizer=tokenizer)
print("✅ 本地 Qwen3-0.6B 已成功接入 LangChain！")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

✅ 本地 Qwen3-0.6B 已成功接入 LangChain！


In [17]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

**构建RAG链**

LCEL写法，全称是 LangChain Expression Language（LangChain 表达式语言）
```
# 像在搭积木，每一步清清楚楚
chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
```

In [19]:
template = """已知信息：
{context}

请根据上面的已知信息回答问题。
问题：{input}
回答："""

prompt = PromptTemplate.from_template(template)

# 构建 Retriever（从向量库检索 Top 3）
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 准备一个辅助函数：把检索到的 Document 对象列表合并成纯字符串
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
    
rag_chain = (
    # 第一步：构造字典，准备填坑的数据
    {
        "context": retriever | format_docs,          # 检索出文档 -> 拼成字符串
        "input": RunnablePassthrough()               # 把用户原始问题原封不动传过来
    }
    # 第二步：自动填坑（相当于自动调用 prompt.format()）
    | prompt
    # 第三步：把填好的提示词丢给你的本地 Qwen 模型
    | qwen3_06b_llm
    # 第四步：解析模型输出，变成干净的字符串
    | StrOutputParser()
)
print("✅ RAG 链构建完成！")

✅ RAG 链构建完成！


**使用语言模型来测试**

In [21]:
query = "Agent中的RAG是什么？"

# 运行RAG链
response = rag_chain.invoke(query)
print("🤖 Qwen3 回答：")
print(response)


🧠 [模型思考]: <think>
好的，我现在要回答用户的问题：“Agent中的RAG是什么？”根据提供的已知信息，我需要仔细分析和整理相关内容。

首先，用户的问题是关于Agent中的RAG（检索-生成）机制。根据已知信息，RAG的核⼼原理是结合“检索”和“生成”，当用户提出查询时，系统首先通过检索模块找到相关文本段落，然后将这些段落作为附加信息传递给语言模型，模型据此生成更准确的回答。同时，RAG能够缓解大模型的“幻觉”问题，因为生成的内容基于真实文档，具有可追溯性和可信度。此外，RAG还加快了知识更新速度，及时反映最新的领域动态。

接下来，我需要将这些信息组织成一个清晰的回答。需要注意的是，用户可能希望得到简明扼要的解释，同时涵盖关键点，如检索和生成的作用、缓解幻觉、知识更新等。同时，要确保回答符合问题的要求，即明确说明RAG在Agent中的具体作用。

在回答时，应该分点说明，但根据用户的问题，可能只需要简要总结。例如，可以指出RAG是Agent的一部分，结合检索和生成，确保回答准确且符合已知信息。
</think>

🤖 Qwen3 回答：
Agent中的RAG（检索-生成）是指将检索和生成结合的机制，具体作用如下：

1. **检索**：系统通过编码器将文档库分割为短片段，并构建向量索引，以便快速定位与问题相关的内容。  
2. **生成**：将检索到的上下文信息作为输入，生成精准且可信的回答，确保内容基于真实文档。  
3. **作用**：缓解大模型的“幻觉”问题，提升回答的可追溯性和可信度；同时加快知识更新速度，及时反映领域动态。  

RAG通过将检索与生成整合，增强了模型对真实信息的依赖，从而优化了回答的准确性与时效性。


看一下上述回答的参考资料

In [23]:
# 也可以顺便看看检索到的参考资料
docs = retriever.invoke(query)
print("\n📚 参考来源：")
for i, doc in enumerate(docs, 1):
    print(f"--- 来源 {i} (第 {doc.metadata.get('page', '?')+1} 页) ---")
    print(doc.page_content[:200] + "...\n")


📚 参考来源：
--- 来源 1 (第 153 页) ---
内容更加符合实时性要求。
RAG 的核⼼原理在于将“检索”与“⽣成”结合：当⽤户提出查询时，系统⾸先通过检索模块找到与问题相关的⽂本⽚
段，然后将这些⽚段作为附加信息传递给语⾔模型，模型据此⽣成更为精准和可靠的回答。通过这种⽅式，RAG 有
效缓解了⼤语⾔模型的“幻觉”问题，因为⽣成的内容建⽴在真实⽂档的基础上，使得答案更具可追溯性和可信度。
同时，由于引⼊了最新的信息源，RAG 技术⼤⼤加快了知...

--- 来源 2 (第 162 页) ---
注：7.2 章节的所有代码均可在 Happy-LLM Chapter7 RAG 中找到。
7.3 Agent  
7.3.1 什么是 LLM Agent？  
简单来说，⼤模型Agent是⼀个以LLM为核⼼“⼤脑”，并赋予其⾃主规划、记忆和使⽤⼯具能⼒的系统。 它不再仅
仅是被动地响应⽤户的提示（Prompt），⽽是能够：
1. 理解⽬标（Goal Understanding）： 接收⼀个相对复杂...

--- 来源 3 (第 154 页) ---
图7.5 TinyRAG 项⽬结构
接下来，让我们梳理⼀下RAG的流程是什么样的呢？
索引：将⽂档库分割成较短的⽚段，并通过编码器构建向量索引。
检索：根据问题和⽚段的相似度检索相关⽂档⽚段。
⽣成：以检索到的上下⽂为条件，⽣成问题的回答。
如下图7.6所示的流程图，图⽚出处 Retrieval-Augmented Generation for Large Language Models: A S...



In [24]:
del model
del tokenizer
CleanMemory()

#### 其他模型写法

**如果使用OpenAI API 模型，可以这样写**
```python
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="xxxx-xx",                  # 换成你想用的模型，比如 gpt-3.5-turbo, gpt-4o, deepseek-chat 等
    base_url="https://xx.xxx.xx",     # 换成你的 API 提供商的 base_url
    api_key="xx-xxxxxxxxxxxxxxxx",    # 换成你的真实 API Key, 本地部署的可以填任意字符串
    temperature=0.1, # 让回答更基于事实
)

# rag_chain = (
#     {
#         "context": retriever | format_docs,   
#         "input": RunnablePassthrough()          
#     }
#     | prompt
#     | llm      <---------------------只改这里
#     | StrOutputParser()
# )
```

**还可以将我们的模型升级成BaseChatModel，无需部署**


In [34]:
from langchain_core.language_models.chat_models import BaseChatModel, ChatResult, ChatGeneration
from langchain_core.messages import AIMessage, BaseMessage
from typing import Optional, List, Any

class QwenChatModel(BaseChatModel):
    model: Any
    tokenizer: Any

    @property
    def _llm_type(self) -> str:
        return "local-qwen3-chat"

    # 🌟 核心修改：接收的是结构化的 messages 列表
    def _generate(self, messages: List[BaseMessage], stop: Optional[List[str]] = None, **kwargs) -> Any:
        
        # 1. 把 LangChain 的 Message 格式转换成我们函数需要的 字典 格式
        my_messages = []
        for msg in messages:
            if msg.type == "system":
                my_messages.append({"role": "system", "content": msg.content})
            elif msg.type == "human":
                my_messages.append({"role": "user", "content": msg.content})
            elif msg.type == "ai":
                my_messages.append({"role": "assistant", "content": msg.content})
        
        # 2. 调用我们的推理函数
        think_output, output = get_qwen_output(my_messages, self.model, self.tokenizer)
        
        # 3. 包装
        # AIMessage
        ai_message = AIMessage(content=output)
        # ChatGeneration
        generation = ChatGeneration(message=ai_message)
        # ChatResult
        return ChatResult(generations=[generation])

In [35]:
model, tokenizer = load_model()
llm = QwenChatModel(model=model, tokenizer=tokenizer)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

In [38]:
template = """已知信息：
{context}

请根据上面的已知信息回答问题。
问题：{input}
回答："""
prompt = PromptTemplate.from_template(template)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
rag_chain = (
    {
        "context": retriever | format_docs,      
        "input": RunnablePassthrough()            
    }
    | prompt
    | llm
    | StrOutputParser()
)
query = "Agent中的RAG是什么？"

# 运行RAG链
response = rag_chain.invoke(query)
print("🤖 Qwen3 回答：")
print(response)

🤖 Qwen3 回答：
Agent中的RAG（Retrieval-Augmented Generation）是指Agent通过检索相关文档或信息，以增强生成内容的准确性和可靠性。具体来说，RAG通过结合检索模块找到与问题相关的文本段落，并将这些段落作为附加信息传递给语言模型，从而生成更精准、可信的回答。这有效缓解了大语言模型的“幻觉”问题，确保回答基于真实数据，同时加快知识更新速度。


In [42]:
del model
del tokenizer
del llm
CleanMemory()

##  a
